# C6 · Sustracción de halo LPM

**Spec:** [`docs/spec_C6_codex_lpm_subtraction.md`](../docs/spec_C6_codex_lpm_subtraction.md)  |  **Bloque:** C · Extracción  |  **Run por defecto:** `ROXs12b_realigned`

Sustrae el halo estelar modelando cada spaxel como modulación polinomial de Legendre (grado 4) del espectro de referencia, con las líneas de ciencia enmascaradas del ajuste (Julo et al. 2025 App. A.4).

| | |
|---|---|
| **Entrada** | `stage02` stack + posiciones (B3) + PSF (C1, solo apcorr) |
| **Salida (QC/productos)** | `stages/spec_lpm_qc.json`, `spec_lpm_object.fits`, cubo residual, mapas de coeficientes |
| **Consume aguas abajo** | D1 v3, G1, E4; cubo residual → E1b (mapa FoV) |


## Qué hace C6 y por qué preserva las líneas

C6 implementa el método **propuesto** por Julo et al. 2025 (LPM): cada spaxel se modela como `ŝ_xy = Σ_k β_k · P_k(λ̃) · ŝ` (Legendre de grado 4 modulando la referencia), resuelto por mínimos cuadrados **con las líneas de ciencia fuera del ajuste**. El modelo interpola suavemente a través de las líneas → el flujo y el perfil del compañero sobreviven (sin auto-sustracción estructural), y el continuo vecino no se hunde.

**Diagnósticos de grado (QC, nunca ajuste al vuelo):** energy-share por grado (Fig. 8 del paper), curva MSE analítica descompuesta en underfit-estrella / overfit-planeta / overfit-ruido (Fig. 6 / Ec. B.5) y mapas de coeficientes que descomponen la PSF por frecuencia espectral (Fig. 7: radio AO, spikes, anillos de Airy). Si señalan que ∂=4 no basta, se revisa la spec — el grado no cambia dentro del run.

Límite conocido (paper §4.1): la componente del espectro planetario **colineal** con la referencia (p.ej. su continuo) se absorbe en el modelo; el LPM es óptimo para compañeras dominadas por líneas.


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs12b_realigned   # o ROXs12b_B_adp para comparar
cd MUSE-accretion-pipeline                    # raíz del repo
python -m musepipe.stages.stage_x05_lpm --run-id $RUN
```

Ligero (~1–2 min: una pseudo-inversa compartida + diagnósticos de grado).

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
RUN_ID = nb.resolve_run_id('ROXs12b_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('stages/spec_lpm_qc.json', RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'python -m musepipe.stages.stage_x05_lpm --run-id $RUN'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('stages/spec_lpm_qc.json', RUN_ID)
nb.show(qc, keys=['lpm.degree', 'lpm.line_preservation_recovery', 'degree_diagnostics.mse_argmin', 'checks.v2_line_preservation_ok', 'checks.v4_slow_path_ok', 'checks.v5_scale_convention_ok'], title='C6')


## Evidencia: preservación de línea y diagnósticos de grado


In [ ]:
if qc is None:
    print('(evidencia omitida: la etapa no se ha ejecutado para esta cadena)')
else:
    with nb.evidence_guard('C6', 'stages/spec_lpm_qc.json'):
        q = nb.load_qc('stages/spec_lpm_qc.json', RUN_ID)
        l = q['lpm']; d = q['degree_diagnostics']
        print(f"grado={l['degree']}  máscara={l['masked_lines_A']}")
        print(f"condición={max(l['condition_number']):.2e}  slow_frac={l['slow_fraction_max']:.3f}")
        print(f"smoke de preservación de línea (control): {l['line_preservation_recovery']}")
        print(f"energy-share (g1..g9): {[f'{v:.3f}' for v in d['energy_share']]}")
        print(f"MSE argmin={d['mse_argmin']}  warns: grado={d['degree_check_warn']} mse={d['mse_check_warn']}")
        print('checks:', q['checks'])


## Plot 1 — mapas de coeficientes (Fig. 7 del paper)

Planos β̂_k del ajuste diagnóstico de grado 9 (primera exposición): los grados bajos muestran el radio AO y los spikes; los altos, anillos de Airy y ruido.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from astropy.io import fits
    q = nb.load_qc('stages/spec_lpm_qc.json', RUN_ID)
    with fits.open(q['products']['coeff_maps']) as h:
        maps = np.asarray(h['COEFFS_DEG9'].data, float)
    fig, axes = plt.subplots(2, 5, figsize=(14, 5.6))
    for k, ax in enumerate(axes.ravel()):
        m = maps[k]
        v = np.nanpercentile(m, [25, 75])
        ax.imshow(m, origin='lower', cmap='RdBu_r', vmin=v[0], vmax=v[1])
        ax.set_title(f'grado {k}', fontsize=9); ax.axis('off')
    fig.suptitle('C6 · mapas de coeficientes LPM (descomposición de la PSF)')
    fig.tight_layout(); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Plot 2 — SGF vs LPM alrededor de Hα (Fig. 13/14 del paper)

Los dos espectros del compañero en ±80 Å de Hα: si hay línea, el SGF la auto-sustrae y hunde el continuo vecino; el LPM la preserva. Requiere C5 corrido.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from astropy.io import fits
    rd = nb.run_dir(RUN_ID)
    def spec(name):
        h = fits.open(rd / 'stages' / name)
        w = np.asarray(h[1].data['wave_A'], float); f = np.asarray(h[1].data['flux'], float)
        h.close(); return w, f
    w_s, f_s = spec('spec_sgf_object.fits')
    w_l, f_l = spec('spec_lpm_object.fits')
    sel = np.abs(w_s - 6563.0) <= 80
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(w_s[sel], f_s[sel], lw=1.0, color='tab:blue', label='SGF (C5)')
    ax.plot(w_l[sel], f_l[sel], lw=1.0, color='tab:orange', label='LPM (C6)')
    ax.axvline(6563, color='tab:red', ls=':'); ax.axhline(0, color='0.6', lw=0.6)
    ax.set_xlabel('λ [Å]'); ax.set_title('C6 · SGF vs LPM alrededor de Hα')
    ax.legend(); fig.tight_layout(); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Decisiones y notas
- **Grado 4 congelado** (tres vías independientes del paper §3.2); diagnósticos de grado como QC con warnings, nunca ajuste al vuelo. · [`docs/spec_C6_codex_lpm_subtraction.md`](../docs/spec_C6_codex_lpm_subtraction.md)
- Máscara de líneas por target (`lpm_masked_lines_A`); una línea de ciencia sin enmascarar = auto-sustracción parcial silenciosa → el notebook la audita contra el catálogo G2.
- Sin pesos por varianza en v1 (OLS plano, como el paper); ponderación por precisión es trabajo futuro explícito. · [`docs/plan_integracion_halosub_julo2025.md`](../docs/plan_integracion_halosub_julo2025.md)


## Checks


In [ ]:
try:
    q = nb.load_qc('stages/spec_lpm_qc.json', RUN_ID)
    for k, v in q['checks'].items():
        print(f'  {k}: {v}')
    assert q['lpm']['degree'] == 4
except FileNotFoundError as e:
    print('QC aún no existe para este run:', e)


## Estado

**Pendiente de primera ejecución sobre datos reales** (checkpoint de la spec C6). Kernel verificado contra los oráculos del paper (Ec. B.5, límite R→0 de Fig. 2d) y contrato de etapa con tests sintéticos (2026-07-14).
